# Train the matched XGBoost predictors

Install `python -m pip install -e .` and
`python -m pip install -e "./training[notebooks]"` from the repository root.
This replaces the old power-loss training notebook inside the runtime
package. See `training/README.md` for the three-channel training workflow.

In [ ]:
from pathlib import Path
import pandas as pd
from pvcracks_training.powerloss.training import train_xgb_models

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "training/pyproject.toml").is_file())
# Set TABLE to your aligned embeddings + morphology + measured targets.
# This small reference table makes the example executable; training on
# these 26 rows does not reproduce the supplied full-dataset model.
TABLE = ROOT / "docs/data/autoencoder_3ch_reference.csv"
OUTPUT = ROOT / "outputs/xgboost_training_example"
table = pd.read_csv(TABLE)
summary = train_xgb_models(table, OUTPUT, seed=30)
display(summary)

Input features are `latent_0` ... `latent_319`, followed by the three
crack/busbar/dark fractions and three 8-connected component counts.
Targets are `delta_pmp_percent`, `delta_voc_percent`, and `Rsh`.
Calculate embeddings and counts using `pvcracks.vae.features.extract_features`.
Do not scale the XGBoost inputs. The saved JSON + metadata pair can be
loaded with the runtime API:

In [ ]:
from pvcracks.powerloss.powerloss_functions import load_all_xgb_models

models = load_all_xgb_models(OUTPUT)
display(pd.DataFrame({target: predictor.predict(table)
                      for target, predictor in models.items()}).head())

Settings match the source notebook: 100 trees, learning rate 0.05,
depth 3, row/column subsampling 0.85, L2 penalty 1, seed 30, and an 80/20
random row split per target. Each target drops rows missing its own
measurement. The export keeps the model fitted to the training split.
Measurements from a shared cell or module can occur in both partitions;
use a grouped split for a separate study of generalization to new modules.